# Experiment 12 — Task-2-pretrained-encoder transfer for CORN wound staging — Colab Run

Task 2's own MAnet+MiT-B2 encoder (`task_2/configs/base.yaml`'s segmentation-benchmarking grid, `paper_miou=0.7263` for this exact architecture+encoder pair) already learned to separate granulation / slough / necrosis / tendon / bone at the pixel level, on a dataset (`datasets/wound-tissue-segmentation/`) with **zero PIID overlap** with the staging dataset task_3 trains on (see `more_experiments/manifests/piid_overlap_report.json`'s `task2_piid_overlap_check`). This experiment discards Task 2's segmentation decoder, keeps only the fine-tuned encoder, attaches a fresh CORN ordinal head, and fully fine-tunes on the PIID staging dataset -- same 3-fold split and same training recipe as task_3's own ablation grid, read directly from `task_3/configs/base.yaml` so there is no hyperparameter drift.

**Fair control:** task_3's own ablation grid already contains an ImageNet-init MiT-B2+CORN row (`mit_b2__corn`, NOT retrained here) -- so the comparison is ImageNet MiT-B2 vs Task2-init MiT-B2, both CORN, both the exact same 3 test folds. MiT-B1+CORN (`mit_b1__corn`) is kept as a model-size reference, also not retrained.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the PIID staging dataset

Same `wound-stage-classification.zip` task_3's own notebook uses -- the ORIGINAL, unmodified RGB photos. No tissue-segmentation data is needed on this side (only Task 2's already-trained *checkpoint*, fetched in step 5).

In [ ]:
import glob

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-stage-classification"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if not os.path.exists(DATASET_DIR):
    os.symlink(candidates[0], DATASET_DIR)
print(DATASET_DIR, "->", os.path.realpath(DATASET_DIR))

## 4. Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/more_experiments/exp/exp12_segmentation_pretrained_transfer/requirements.txt

## 5. Fetch Task 2's MAnet+MiT-B2 encoder checkpoints (fold 1/2/3)

These are the ONLY 3 files needed out of Task 2's full (12GB) checkpoint directory -- ~140MB each. Expected at `drive/MyDrive/thesis/task_2_outputs/checkpoints/wound_inner_fold{1,2,3}/MAnet_mit_b2_imagenet.pth` (same Drive layout convention task_3's own notebook uses for its checkpoints). Upload these 3 files there before running this cell if they aren't already.

In [ ]:
import shutil

os.chdir(f"{REPO_DIR}/task_2")
DRIVE_TASK2_DIR = "/content/drive/MyDrive/thesis/task_2_outputs/checkpoints"

for task2_fold in [1, 2, 3]:
    local_dir = f"results/task_2_outputs/checkpoints/wound_inner_fold{task2_fold}"
    local_path = f"{local_dir}/MAnet_mit_b2_imagenet.pth"
    os.makedirs(local_dir, exist_ok=True)
    if os.path.exists(local_path):
        print(f"fold{task2_fold}: already present locally")
        continue
    drive_path = f"{DRIVE_TASK2_DIR}/wound_inner_fold{task2_fold}/MAnet_mit_b2_imagenet.pth"
    assert os.path.exists(drive_path), f"not found on Drive: {drive_path} -- upload it first"
    shutil.copy(drive_path, local_path)
    print(f"fold{task2_fold}: copied from Drive ({os.path.getsize(local_path) / 1e6:.0f} MB)")

## 6. Architecture sanity check

No training needed -- verifies Task 2's checkpoint loads into task_3's own MiT-B2 `Backbone` with a fully matched key set (332/332), weights actually change from random init, and the CORN head's output shape is correct.

In [ ]:
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp12_segmentation_pretrained_transfer/scripts")
!python test_model.py

## 7. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp12_segmentation_pretrained_transfer_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/exp12_results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 8. Train — all 3 folds

Each fold loads its own matching Task 2 encoder checkpoint (fold `f` -> Task 2's `wound_inner_fold{f+1}`, see README for why the pairing is arbitrary-but-fixed), verifies the encoder weight transfer before training (stops immediately if the key set doesn't fully match or weights didn't change), then fully fine-tunes encoder+head together -- no freezing, same recipe as task_3's own ablation grid. Safe to interrupt and re-run; already-completed folds are skipped.

In [ ]:
!python train.py --workers 4

## 9. Compare against the ImageNet-init references

`mit_b1__corn` and `mit_b2__corn` are read directly from task_3's OWN `ablation_results.csv` (not retrained here, not modified). `mit_b2_task2init__corn` is this run's own results.

In [ ]:
!python compare_results.py